# Pooled clinical-field organization with direct Oλ
Use 500 Bayesian correlation draws and 10,000 within-dataset membership permutations per context and representation. Both geometries use dataset-adjusted draw-specific correlations, averaged arithmetically. Each context uses its own adjusted midpoint survival fit to define adverse and favorable regimes.

The initial queue contains direct Oλ, ten TCAT GEPs and three hard-label atlas cell types ranked by absolute log median continuous HR per SD, plus antigen-associated activation (ASA). Continuous effects select the initial queue only. A feature becomes runnable as soon as its individual transition fit finishes. Rerun the selection and analysis cells to collect newly completed fits. Enable ADD_TRANSITION_STRONG to append every eligible midpoint predictor whose 95% HR interval excludes 1. Rankings and multiple-testing corrections apply to the currently completed family.

Drive retains the observed matrices, clinical vectors, correlation reconstruction inputs, PSD diagnostics, and permutation nulls. Interrupted permutation jobs resume from their last 100-permutation checkpoint.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile
import numpy as np
import pandas as pd

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')

import os
from IPython.display import display
DATA_DIR = DATA

TCAT = DATA/'patient_TCAT_particles'
ATLAS = DATA/'Functional_CART_atlas_patient_particles_hard_labels_all8'
O = DATA/'olambda_reciprocal_direct_no_asymptote_floor_v1'
oe, op = O/'patient_olambda_asymptotes.csv', O/'patient_olambda_particles.csv'
outcomes = DATA/'clinical_outcomes.csv'
inputs = {
    'gep': (TCAT/'all_patient_TCAT_estimates_harmonized.csv', TCAT/'all_patient_TCAT_particles_harmonized.npz'),
    'atlas': (ATLAS/'patient_atlas_estimates.csv', ATLAS/'patient_atlas_particles.npz'),
}
O_ROOT = DATA/'bayesian_Olambda_survival'/'direct_reciprocal_adjusted_midpoint_v1'/'smooth_transition_Q10_Q90_midpoint'
SURVIVAL_OUTPUT = DATA/'direct_reciprocal_TCAT_atlas_survival_v1'
draws, tune, chains, cores = 2000, 2000, 4, 4
width, target_accept, seed = 0.10, 0.99, 0
adjustments = {'Treatment':'axicel','Disease':'lymphoma'}

OLAMBDA_TRANSITION_CSV = None
ADD_TRANSITION_STRONG = False
EXTRA_FEATURES = []  # Optional exact GEP, score or atlas feature names.

In [ ]:
TCAT_PARTICLES = DATA_DIR / 'patient_TCAT_particles' / 'all_patient_TCAT_particles_harmonized.npz'
ATLAS_PARTICLES = (DATA_DIR / 'Functional_CART_atlas_patient_particles_hard_labels_all8' /
                   'patient_atlas_particles.npz')

TCAT_SURVIVAL = DATA_DIR / 'bayesian_TCAT_survival' / 'adjusted'
TCAT_CONTINUOUS = TCAT_SURVIVAL / 'individual_all' / 'individual_results.csv'
ATLAS_SURVIVAL = (DATA_DIR / 'bayesian_Functional_CART_atlas_survival' /
                  'adjusted_hard_labels_all8')
ATLAS_CONTINUOUS = ATLAS_SURVIVAL / 'continuous' / 'individual_results.csv'

CORRELATION_DRAWS, PERMUTATIONS, RANDOM_STATE = 500, 10000, 0
OUTPUT_DIR = DATA / 'predictive_field_direct_midpoint_mean_dataset_adjusted_v1'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
REPRESENTATIONS = {
    'TCAT_GEP': {
        'particles': TCAT_PARTICLES, 'continuous': TCAT_CONTINUOUS,
        'prefix': 'TCAT_GEP_',
        'feature_transform': ('cell-level Bayesian-bootstrap-weighted mean of positive TCAT GEP compositions; '
                              'CLR_j = log(x_j) - mean_k(log(x_k)); no GEP pseudocount added'),
        'compositional_constraint': 'sum of TCAT GEP CLR coordinates equals zero before correlation standardization',
    },
    'atlas_cell_type': {
        'particles': ATLAS_PARTICLES, 'continuous': ATLAS_CONTINUOUS,
        'prefix': 'ATLAS_proportion_',
        'feature_transform': ('cell-level Bayesian-bootstrap-weighted hard-label counts multiplied by n_cells; '
                              'add 0.5 to every cell-type count, normalize to proportions, then '
                              'CLR_j = log(p_j) - mean_k(log(p_k))'),
        'compositional_constraint': 'sum of atlas cell-type CLR coordinates equals zero before correlation standardization',
    },
}

In [ ]:
def digest(path):
    with open(path,'rb') as stream:
        h = hashlib.sha256()
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
        return h.hexdigest()


from functools import lru_cache
file_hash = lru_cache(maxsize=None)(digest)
common = ['--minimum-cells','1000','--draws',str(draws),'--tune',str(tune),'--chains',str(chains),
          '--cores',str(cores),'--target-accept',str(target_accept),'--random-state',str(seed),
          '--adjustment','Treatment=axicel','--adjustment','Disease=lymphoma']
midpoint = ['--midpoint-centered','--transition-scale-sd',str(width)]
quantiles = ['--crossover-quantile-low','0.10','--crossover-quantile-high','0.90']


# Resolve the exact hash directories used by the production individual midpoint notebook.
# Aggregate result tables are written later than each finished feature.
def individual_dir(name, names):
    flags = [*common,*midpoint,*quantiles]
    for f in names:
        flags.extend(['--feature',f])
    spec = dict(revision='8a4179d5d133c9e04713420f4ea7ea2768e8b1a0',
                inputs=[file_hash(p) for p in [*inputs[name],outcomes]], flags=flags)
    tag = hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()[:12]
    return SURVIVAL_OUTPUT/name/'individual_midpoint'/tag/'individual'
def olambda_crossover():
    paths = [Path(OLAMBDA_TRANSITION_CSV)] if OLAMBDA_TRANSITION_CSV else sorted(O_ROOT.glob('*/model_result.csv'))
    valid = []
    for path in paths:
        root = path.parent
        if '-running-' in root.name: continue
        if not (root/'notebook_config.json').is_file() or not (root/'run_manifest.json').is_file():
            continue
        config = json.loads((root/'notebook_config.json').read_text())
        manifest = json.loads((root/'run_manifest.json').read_text())
        prior = manifest.get('crossover_prior',{})
        if (config['input_sha256'] == {p.name:file_hash(p) for p in (oe,op,outcomes)}
            and manifest.get('hazard_activation_offset')==0.5 and manifest.get('adjustments')==adjustments
            and manifest.get('transition_scale_sd')==width
            and (prior.get('quantile_low'),prior.get('quantile_high'))==(0.10,0.90)):
            result = pd.read_csv(path)
            if len(result)==1 and 0 < result.crossover_median.iloc[0] < 1 and result.patients.iloc[0]==163:
                valid.append(path)
    if not valid and not OLAMBDA_TRANSITION_CSV:
        return None
    if len(valid)!=1:
        raise ValueError(f'Expected one matching reciprocal midpoint Oλ run; found {valid}. Set OLAMBDA_TRANSITION_CSV or finish the Oλ survival notebook.')
    return valid[0]

In [ ]:
METRICS = ['prognostic_channeling', 'signed_graph_variation', 'signed_neighbor_cosine']
DESIRED_SIGN = np.asarray([1.0, -1.0, 1.0])
PSD_TOLERANCE = -1e-10


def safe_name(value):
    return ''.join(character if character.isalnum() or character in '._-' else '_' for character in str(value))


def weighted_correlation_and_scale(values, weights):
    weights = np.asarray(weights, dtype=float)
    total = weights.sum()
    if not np.isfinite(total) or total <= 0:
        raise ValueError('Correlation weights have no positive finite mass.')
    weights = weights / total
    centered = values - np.sum(weights[:, None] * values, axis=0)
    covariance = (centered * weights[:, None]).T @ centered
    scale = np.sqrt(np.maximum(np.diag(covariance), 1e-12))
    correlation = covariance / scale[:, None] / scale[None, :]
    correlation = (correlation + correlation.T) / 2
    np.fill_diagonal(correlation, 1)
    return correlation, scale


def weighted_correlation(values, weights):
    return weighted_correlation_and_scale(values, weights)[0]


def dataset_residuals(values, weights, datasets):
    residuals = np.asarray(values, dtype=float).copy()
    for dataset in np.unique(datasets):
        keep = datasets == dataset
        group_weights = weights[keep]
        group_weights = group_weights / group_weights.sum()
        residuals[keep] -= np.sum(values[keep] * group_weights[:, None], axis=0)
    return residuals


def posterior_states(values, particle_weights, datasets, draws, random_state):
    rng = np.random.default_rng(random_state)
    patients, particles, features = values.shape
    adjusted = np.empty((draws, patients, features), dtype=np.float32)
    bootstrap_weights = np.empty((draws, patients), dtype=np.float32)
    for draw in range(draws):
        particle_index = np.asarray([
            rng.choice(particles, p=particle_weights[patient])
            for patient in range(patients)
        ])
        weights = rng.dirichlet(np.ones(patients))
        state = values[np.arange(patients), particle_index]
        adjusted[draw] = dataset_residuals(state, weights, datasets)
        bootstrap_weights[draw] = weights
    return adjusted, bootstrap_weights


def mean_correlations(states, bootstrap_weights, membership, indices, feature_indices=None,
                      return_scales=False):
    if feature_indices is None:
        feature_indices = np.arange(states.shape[2])
    adverse_sum = np.zeros((len(feature_indices), len(feature_indices)), dtype=float)
    favorable_sum = np.zeros_like(adverse_sum)
    adverse_scales = np.empty((states.shape[0], len(feature_indices)), dtype=np.float32)
    favorable_scales = np.empty_like(adverse_scales)
    membership = np.asarray(membership, dtype=float)
    for draw in range(states.shape[0]):
        values = states[draw, indices][:, feature_indices]
        weights = bootstrap_weights[draw, indices]
        adverse_correlation, adverse_scale = weighted_correlation_and_scale(
            values, weights * membership
        )
        favorable_correlation, favorable_scale = weighted_correlation_and_scale(
            values, weights * (1 - membership)
        )
        adverse_sum += adverse_correlation
        favorable_sum += favorable_correlation
        adverse_scales[draw] = adverse_scale
        favorable_scales[draw] = favorable_scale
    adverse = adverse_sum / states.shape[0]
    favorable = favorable_sum / states.shape[0]
    if return_scales:
        return adverse, favorable, adverse_scales, favorable_scales
    return adverse, favorable


def save_reproduction_inputs(path, states, bootstrap_weights, datasets, sample_ids,
                             features, metadata):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        dataset_adjusted_state_draws=np.asarray(states, dtype=np.float32),
        bayesian_bootstrap_weights=np.asarray(bootstrap_weights, dtype=np.float32),
        datasets=np.asarray(datasets, dtype='U'),
        sample_ids=np.asarray(sample_ids, dtype='U'),
        feature_names=np.asarray(features, dtype='U'),
        metadata_json=np.asarray(json.dumps(metadata, sort_keys=True)),
    )


def save_field_information(path, adverse, favorable, adverse_scales, favorable_scales,
                           beta, features, raw_membership, adverse_membership,
                           patient_indices, removed_coordinates, reproduction_inputs,
                           coefficient_centers, coefficient_scales, adverse_high, metadata):
    path.parent.mkdir(parents=True, exist_ok=True)
    high = adverse if adverse_high else favorable
    low = favorable if adverse_high else adverse
    high_scales = adverse_scales if adverse_high else favorable_scales
    low_scales = favorable_scales if adverse_high else adverse_scales
    np.savez_compressed(
        path,
        high_mean_correlation=np.asarray(high, dtype=float),
        low_mean_correlation=np.asarray(low, dtype=float),
        adverse_mean_correlation=np.asarray(adverse, dtype=float),
        favorable_mean_correlation=np.asarray(favorable, dtype=float),
        high_feature_sd_by_draw=np.asarray(high_scales, dtype=np.float32),
        low_feature_sd_by_draw=np.asarray(low_scales, dtype=np.float32),
        adverse_feature_sd_by_draw=np.asarray(adverse_scales, dtype=np.float32),
        favorable_feature_sd_by_draw=np.asarray(favorable_scales, dtype=np.float32),
        clinical_beta=np.asarray(beta, dtype=float),
        feature_names=np.asarray(features, dtype='U'),
        raw_probability_high=np.asarray(raw_membership, dtype=float),
        adverse_regime_membership=np.asarray(adverse_membership, dtype=float),
        patient_indices=np.asarray(patient_indices, dtype=int),
        removed_coordinates=np.asarray(removed_coordinates, dtype='U'),
        coefficient_feature_centers=np.asarray(coefficient_centers, dtype=float),
        coefficient_feature_scales=np.asarray(coefficient_scales, dtype=float),
        adverse_high_on_original_predictor=np.asarray(bool(adverse_high)),
        reproduction_inputs_path=np.asarray(str(reproduction_inputs)),
        metadata_json=np.asarray(json.dumps(metadata, sort_keys=True)),
    )


def save_psd_failure_bundle(path, states, bootstrap_weights, membership, indices,
                            feature_indices, adverse_mean, favorable_mean):
    """Retain the actual input matrices if a mean unexpectedly fails PSD."""
    if feature_indices is None:
        feature_indices = np.arange(states.shape[2])
    adverse_draws = np.empty(
        (states.shape[0], len(feature_indices), len(feature_indices)), dtype=np.float32
    )
    favorable_draws = np.empty_like(adverse_draws)
    for draw in range(states.shape[0]):
        values = states[draw, indices][:, feature_indices]
        weights = bootstrap_weights[draw, indices]
        adverse_draws[draw] = weighted_correlation(values, weights * membership)
        favorable_draws[draw] = weighted_correlation(values, weights * (1 - membership))
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path, adverse_draws=adverse_draws, favorable_draws=favorable_draws,
        adverse_mean=adverse_mean, favorable_mean=favorable_mean,
        membership=np.asarray(membership, dtype=float),
    )


def field_metrics(correlation, beta):
    beta = np.asarray(beta, dtype=float)
    first, second = np.triu_indices(len(beta), 1)
    edge = correlation[first, second]
    weight = edge ** 2
    sign = np.where(edge < 0, -1.0, 1.0)
    degree = np.zeros(len(beta))
    np.add.at(degree, first, weight)
    np.add.at(degree, second, weight)
    adjacency = np.zeros_like(correlation)
    adjacency[first, second] = weight * sign
    adjacency[second, first] = weight * sign
    neighbor = adjacency @ beta / np.maximum(degree, 1e-12)
    beta_norm = np.maximum(beta @ beta, 1e-12)
    variation_norm = np.maximum(np.sum(degree * beta ** 2), 1e-12)
    cosine_norm = np.sqrt(
        np.maximum(np.sum(degree * beta ** 2), 1e-12)
        * np.maximum(np.sum(degree * neighbor ** 2), 1e-12)
    )
    return np.asarray([
        beta @ correlation @ beta / beta_norm,
        np.sum(weight * (beta[first] - sign * beta[second]) ** 2) / variation_norm,
        np.sum(degree * beta * neighbor) / cosine_norm,
    ])


def field_contrast(adverse, favorable, beta):
    return field_metrics(adverse, beta) - field_metrics(favorable, beta)


def permute_within_dataset(values, groups, rng):
    permuted = np.asarray(values, dtype=float).copy()
    for group in groups:
        permuted[group] = permuted[rng.permutation(group)]
    return permuted


def permutation_summary(observed, null):
    null_mean = null.mean(axis=0)
    null_sd = np.maximum(null.std(axis=0, ddof=1), 1e-12)
    z = (observed - null_mean) / null_sd
    p = (1 + np.sum(null * DESIRED_SIGN >= observed * DESIRED_SIGN, axis=0)) / (len(null) + 1)
    oriented_z = z * DESIRED_SIGN
    joint_z = oriented_z.mean()
    joint_null = ((null - null_mean) / null_sd * DESIRED_SIGN).mean(axis=1)
    joint_p = (1 + np.sum(joint_null >= joint_z)) / (len(null) + 1)
    return z, p, joint_z, joint_p, joint_null


def matrix_diagnostics(matrix):
    matrix = np.asarray(matrix, dtype=float)
    symmetry_error = float(np.max(np.abs(matrix - matrix.T)))
    diagonal_deviation = float(np.max(np.abs(np.diag(matrix) - 1)))
    eigenvalues = np.linalg.eigvalsh((matrix + matrix.T) / 2)
    return np.asarray([
        eigenvalues.min(), np.sum(eigenvalues < PSD_TOLERANCE),
        symmetry_error, diagonal_deviation,
    ], dtype=float)


def diagnostic_record(scope, representation, context, regime, matrix, **extra):
    minimum, negative, symmetry, diagonal = matrix_diagnostics(matrix)
    return {
        'scope': scope, 'representation': representation, 'context': context,
        'regime': regime, 'minimum_eigenvalue': minimum,
        'eigenvalues_below_minus_1e_minus_10': int(negative),
        'maximum_symmetry_error': symmetry,
        'maximum_diagonal_deviation': diagonal, **extra,
    }


def save_permutation_diagnostics(path, diagnostics):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        adverse_minimum_eigenvalue=diagnostics[:, 0, 0],
        adverse_negative_eigenvalues=diagnostics[:, 0, 1].astype(int),
        adverse_symmetry_error=diagnostics[:, 0, 2],
        adverse_diagonal_deviation=diagnostics[:, 0, 3],
        favorable_minimum_eigenvalue=diagnostics[:, 1, 0],
        favorable_negative_eigenvalues=diagnostics[:, 1, 1].astype(int),
        favorable_symmetry_error=diagnostics[:, 1, 2],
        favorable_diagonal_deviation=diagnostics[:, 1, 3],
    )


def diagnostic_summary(scope, representation, context, diagnostics, **extra):
    return {
        'scope': scope, 'representation': representation, 'context': context,
        'permutations': len(diagnostics),
        'minimum_eigenvalue_across_permutations': float(diagnostics[:, :, 0].min()),
        'matrices_failing_psd_tolerance': int(np.sum(diagnostics[:, :, 1] > 0)),
        'maximum_negative_eigenvalue_count': int(diagnostics[:, :, 1].max()),
        'maximum_symmetry_error': float(diagnostics[:, :, 2].max()),
        'maximum_diagonal_deviation': float(diagnostics[:, :, 3].max()),
        **extra,
    }


def benjamini_hochberg(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values)
    adjusted = values[order] * len(values) / np.arange(1, len(values) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    answer = np.empty(len(values), dtype=float)
    answer[order] = np.minimum(adjusted, 1)
    return answer

## Discover completed transitions and build the queue
Rerun from here when more survival fits finish. The queue CSV records the reason for selection and any pending memberships.

In [ ]:
scores = ['TCAT_SCORE_ASA_CLR','TCAT_SCORE_Proliferation_CLR']
CONTEXT_EXCLUDED_FEATURES = {
    scores[0]: ['TCAT_GEP_OX40_EBI3_CLR','TCAT_GEP_ICOS_CD38_CLR',
                'TCAT_GEP_CTLA4_CD38_CLR','TCAT_GEP_TIMD4_TIM3_CLR'],
    scores[1]: ['TCAT_GEP_CellCycle_G2M_CLR','TCAT_GEP_CellCycle_Late_S_CLR','TCAT_GEP_CellCycle_S_CLR'],
}
keys=['Dataset','sample_id']
cohort=pd.read_csv(oe).set_index(keys,verify_integrity=True)
assert len(cohort)==163 and cohort.index.get_level_values(0).nunique()==8
assert cohort.olambda_scale.eq('reciprocal').all() and cohort.model.eq('bounded_direct_no_asymptote_floor').all()
targets={'O_lambda':['system']}
ready={}
o_result=olambda_crossover()
if o_result is not None:
    ready['O_lambda']=(pd.read_csv(o_result).iloc[0],o_result.parent/'patient_regime_probabilities.csv','system')
allowed=set()
for name,field,n_top in [('gep','TCAT_GEP',10),('atlas','atlas_cell_type',3)]:
    spec=REPRESENTATIONS[field]
    with np.load(spec['particles']) as archive:
        nodes=[f for f in archive['feature_names'].astype(str) if f.startswith(spec['prefix'])]
    assert len(set(nodes))==len(nodes)==(52 if name=='gep' else 11)
    names=nodes+(scores if name=='gep' else [])
    allowed.update(names)
    table=pd.read_csv(spec['continuous']).set_index('feature',verify_integrity=True).loc[nodes]
    assert np.isfinite(table.HR_median).all() and table.HR_median.gt(0).all()
    top=table.assign(strength=np.log(table.HR_median).abs()).sort_index().sort_values(
        'strength',ascending=False,kind='stable').head(n_top)
    for f in top.index: targets[f]=['continuous top '+str(n_top)]
    if name=='gep': targets[scores[0]]=['requested ASA']
    root=individual_dir(name,names)
    for marker in sorted(root.glob('*/model_manifest.json')):
        m=json.loads(marker.read_text())
        f=m['feature']
        assert f in names and f not in ready and m['hazard_activation_offset']==.5
        directory=marker.parent
        r=pd.read_csv(directory/'model_result.csv').iloc[0]
        assert r.feature==f and r.patients==163 and r.datasets==8 and m['adjustments']==adjustments
        ready[f]=(r,directory/'patient_regime_probabilities.csv','score' if f in scores else field)
        if ADD_TRANSITION_STRONG and (r.HR_q025>1 or r.HR_q975<1):
            targets.setdefault(f,[]).append('strong transition')
if set(EXTRA_FEATURES)-allowed: raise ValueError('Unknown extra features: '+str(set(EXTRA_FEATURES)-allowed))
for f in EXTRA_FEATURES: targets.setdefault(f,[]).append('explicit')
contexts=[]
queue=[]
for f,reasons in targets.items():
    row=dict(context=f,selection='; '.join(reasons),status='ready' if f in ready else 'pending transition')
    queue.append(row)
    if f not in ready: continue
    r,path,kind=ready[f]
    q=pd.read_csv(path).set_index(keys,verify_integrity=True).posterior_mean_probability_high
    assert q.index.sort_values().equals(cohort.index.sort_values()) and q.between(0,1).all()
    contexts.append(dict(context=f,context_kind=kind,selection=row['selection'],membership_path=path,
                         HR_median=float(r.HR_median),HR_q025=float(r.HR_q025),HR_q975=float(r.HR_q975),
                         adverse_high=float(r.HR_median)>=1))
queue=pd.DataFrame(queue)
queue.to_csv(OUTPUT_DIR/'selected_contexts.csv',index=False)
display(queue)

In [ ]:
def load_representation(specification):
    archive = np.load(specification['particles'], allow_pickle=False)
    all_features = archive['feature_names'].astype(str)
    features = np.asarray([feature for feature in all_features if feature.startswith(specification['prefix'])])
    lookup = {feature: index for index, feature in enumerate(all_features)}
    feature_indices = np.asarray([lookup[feature] for feature in features])
    patients = pd.DataFrame({
        'Dataset': archive['datasets'].astype(str),
        'sample_id': archive['patient_ids'].astype(str),
        'particle_row': np.arange(len(archive['patient_ids'])),
    })
    for number, context in enumerate(contexts):
        column = f'membership_{number:03d}'
        membership = pd.read_csv(context['membership_path'])[[
            'Dataset', 'sample_id', 'posterior_mean_probability_high'
        ]].rename(columns={'posterior_mean_probability_high': column})
        patients = patients.merge(membership, on=['Dataset', 'sample_id'], how='inner', validate='one_to_one')
        context['column'] = column
    patients = patients.sort_values(['Dataset', 'sample_id']).reset_index(drop=True)
    assert pd.MultiIndex.from_frame(patients[keys]).equals(cohort.index.sort_values())
    rows = patients.particle_row.to_numpy(int)
    values = np.asarray(archive['values'][rows][:, :, feature_indices], dtype=float)
    particle_weights = np.asarray(archive['particle_weights'][rows], dtype=float)
    particle_weights /= particle_weights.sum(axis=1, keepdims=True)
    effects = pd.read_csv(specification['continuous']).set_index('feature').loc[features]
    beta = np.log(effects.HR_median.to_numpy(float))
    return patients, features, values, particle_weights, beta, effects


if not contexts:
    raise RuntimeError('No selected midpoint fits are complete yet. Rerun the discovery and analysis cells once one finishes.')

result_rows = []
observed_diagnostic_rows = []
permutation_diagnostic_rows = []
matrix_root = OUTPUT_DIR / 'observed_mean_matrices'
diagnostic_root = OUTPUT_DIR / 'permutation_psd_diagnostics'
null_root = OUTPUT_DIR / 'permutation_nulls'
bundle_root = OUTPUT_DIR / 'compositional_sensitivity_bundles'
input_root = OUTPUT_DIR / 'correlation_reproduction_inputs'
for directory in [matrix_root, diagnostic_root, null_root, bundle_root, input_root]:
    directory.mkdir(exist_ok=True)

for representation_number, (representation, specification) in enumerate(REPRESENTATIONS.items()):
    print('\n' + '#' * 110)
    print('REPRESENTATION:', representation)
    print('#' * 110, flush=True)
    patients, features, values, particle_weights, beta, effects = load_representation(specification)
    datasets = patients.Dataset.astype(str).to_numpy()
    groups = [np.flatnonzero(datasets == dataset) for dataset in np.unique(datasets)]
    states, bootstrap_weights = posterior_states(
        values, particle_weights, datasets, CORRELATION_DRAWS, RANDOM_STATE
    )
    reproduction_inputs = input_root / f'{representation}.npz'
    save_reproduction_inputs(
        reproduction_inputs, states, bootstrap_weights, datasets,
        patients.sample_id.astype(str).to_numpy(), features,
        {
            'representation': representation,
            'feature_transform': specification['feature_transform'],
            'compositional_constraint': specification['compositional_constraint'],
            'dataset_adjustment': 'draw-specific Bayesian-bootstrap weighted dataset-mean subtraction',
            'correlation_scaling': 'draw- and regime-specific weighted population SD',
            'correlation_draws': CORRELATION_DRAWS,
            'random_state': RANDOM_STATE,
            'draw_specific_correlation_matrices_saved': False,
            'reproduction': 'use saved state draws, bootstrap weights, context membership, and ordered feature indices',
        },
    )
    all_indices = np.arange(len(patients))
    representation_matrix_dir = matrix_root / representation
    representation_matrix_dir.mkdir(exist_ok=True)
    print(f'{len(patients)} patients | {len(features)} field nodes | {len(contexts)} contexts')

    for context_number, context in enumerate(contexts, start=1):
        result_path = OUTPUT_DIR / f"checkpoint__{representation}__{safe_name(context['context'])}.json"
        bundle_path = bundle_root / representation / f"{safe_name(context['context'])}.npz"
        excluded_features = set(CONTEXT_EXCLUDED_FEATURES.get(context['context'], []))
        if context['context'] in set(features):
            excluded_features.add(context['context'])
        if representation=='TCAT_GEP' and context['context'] in CONTEXT_EXCLUDED_FEATURES:
            assert excluded_features.issubset(features)
        excluded_features &= set(features)
        seed = 100 if context['context']=='O_lambda' else int.from_bytes(
            hashlib.sha256(context['context'].encode()).digest()[:4],'little')
        signature = dict(files=[file_hash(p) for p in (specification['particles'],specification['continuous'])],
                         membership_hash=digest(context['membership_path']), transition_HR=context['HR_median'],
                         interval=[context['HR_q025'],context['HR_q975']], seed=seed,
                         state_seed=RANDOM_STATE, draws=CORRELATION_DRAWS, permutations=PERMUTATIONS,
                         removed=sorted(excluded_features))
        bundle_is_current = False
        if bundle_path.exists():
            saved_removed = set(
                np.load(bundle_path, allow_pickle=False)['removed_coordinates'].astype(str)
            )
            bundle_is_current = excluded_features == saved_removed
        if (result_path.exists() and bundle_is_current and
            json.loads(result_path.read_text()).get('signature') == signature):
            saved = json.loads(result_path.read_text())
            result_rows.append(saved['result'])
            observed_diagnostic_rows.extend(saved['observed_diagnostics'])
            permutation_diagnostic_rows.append(saved['permutation_diagnostic_summary'])
            print(f"[{context_number}/{len(contexts)}] RESUME {context['context']}", flush=True)
            continue

        graph_indices = np.flatnonzero(~np.isin(features, list(excluded_features)))
        graph_features = features[graph_indices]
        graph_beta = beta[graph_indices]
        graph_effects = effects.iloc[graph_indices]
        raw_membership = patients[context['column']].to_numpy(float)
        adverse_membership = raw_membership if context['adverse_high'] else 1 - raw_membership
        adverse, favorable, adverse_scales, favorable_scales = mean_correlations(
            states, bootstrap_weights, adverse_membership, all_indices, graph_indices,
            return_scales=True,
        )
        observed = field_contrast(adverse, favorable, graph_beta)
        observed_diagnostics = [
            diagnostic_record('pooled_observed', representation, context['context'], 'adverse', adverse),
            diagnostic_record('pooled_observed', representation, context['context'], 'favorable', favorable),
        ]
        for diagnostic in observed_diagnostics:
            if diagnostic['eigenvalues_below_minus_1e_minus_10']:
                failure_path = OUTPUT_DIR / 'psd_failures' / f'{representation}__{safe_name(context["context"])}__observed.npz'
                save_psd_failure_bundle(
                    failure_path, states, bootstrap_weights, adverse_membership,
                    all_indices, graph_indices, adverse, favorable,
                )
                raise RuntimeError(f'Mean matrix failed PSD audit; inputs saved to {failure_path}: {diagnostic}')

        removed_coordinates = [feature for feature in features if feature not in set(graph_features)]
        information_metadata = {
            'scope': 'pooled', 'representation': representation,
            'context': context['context'], 'geometry': 'dataset_adjusted',
            'aggregation': 'arithmetic mean of draw-specific correlation matrices',
            'feature_transform': specification['feature_transform'],
            'compositional_constraint': specification['compositional_constraint'],
            'dataset_adjustment': 'draw-specific Bayesian-bootstrap weighted dataset-mean subtraction',
            'correlation_scaling': 'weighted population SD calculated separately for every draw and regime',
            'clinical_coefficients': 'posterior-median log HR per global feature SD',
            'clinical_vector_centering': 'none', 'clinical_vector_projection': 'none',
            'surrogate_centering': 'none', 'surrogate_projection': 'none',
            'draw_specific_correlation_matrices_saved': False,
            'reproduction_inputs_saved': True,
            'random_state': RANDOM_STATE,
            'membership_permutation_seed': seed,
            'membership_permutation_rule': 'permute adverse-regime membership within each dataset',
        }
        save_field_information(
            bundle_path, adverse, favorable, adverse_scales, favorable_scales,
            graph_beta, graph_features, raw_membership, adverse_membership,
            all_indices, removed_coordinates, reproduction_inputs,
            graph_effects['center'].to_numpy(float) if 'center' in graph_effects else np.full(len(graph_features), np.nan),
            graph_effects['scale'].to_numpy(float) if 'scale' in graph_effects else np.full(len(graph_features), np.nan),
            context['adverse_high'],
            information_metadata,
        )

        # Match the established context-specific permutation stream where possible.
        rng = np.random.default_rng(seed)
        null = np.empty((PERMUTATIONS, len(METRICS)), dtype=float)
        permutation_diagnostics = np.empty((PERMUTATIONS, 2, 4), dtype=float)
        partial = result_path.with_suffix('.partial.npz')
        start = 0
        if partial.exists():
            with np.load(partial) as saved:
                if json.loads(str(saved['signature'])) == signature:
                    start = len(saved['null'])
                    null[:start] = saved['null']
                    permutation_diagnostics[:start] = saved['diagnostics']
                    rng.bit_generator.state = json.loads(str(saved['rng']))
                    print('Resuming from permutation',start,flush=True)
        for permutation in range(start,PERMUTATIONS):
            permuted = permute_within_dataset(adverse_membership, groups, rng)
            permuted_adverse, permuted_favorable = mean_correlations(
                states, bootstrap_weights, permuted, all_indices, graph_indices
            )
            null[permutation] = field_contrast(permuted_adverse, permuted_favorable, graph_beta)
            permutation_diagnostics[permutation, 0] = matrix_diagnostics(permuted_adverse)
            permutation_diagnostics[permutation, 1] = matrix_diagnostics(permuted_favorable)
            if np.any(permutation_diagnostics[permutation, :, 1] > 0):
                failure_path = (
                    OUTPUT_DIR / 'psd_failures' /
                    f'{representation}__{safe_name(context["context"])}__permutation_{permutation + 1}.npz'
                )
                save_psd_failure_bundle(
                    failure_path, states, bootstrap_weights, permuted,
                    all_indices, graph_indices, permuted_adverse, permuted_favorable,
                )
                raise RuntimeError(
                    f'Permutation mean matrix failed PSD audit; inputs saved to {failure_path}'
                )
            if (permutation + 1) % 100 == 0:
                tmp = partial.with_suffix('.tmp.npz')
                np.savez(tmp,signature=json.dumps(signature),null=null[:permutation+1],
                         diagnostics=permutation_diagnostics[:permutation+1],rng=json.dumps(rng.bit_generator.state))
                tmp.replace(partial)
                print(f'    permutations {permutation + 1:,}/{PERMUTATIONS:,}', flush=True)

        z, p, joint_z, joint_p, joint_null = permutation_summary(observed, null)
        row = {
            'field': representation, 'context': context['context'],
            'context_kind': context['context_kind'], 'geometry': 'dataset_adjusted',
            'aggregation': 'arithmetic_mean', 'patients': len(patients),
            'graph_nodes': len(graph_features),
            'context_node_excluded': context['context'] in set(features),
            'clinical_adverse_tail': 'high' if context['adverse_high'] else 'low',
            'effective_adverse_patients': float(adverse_membership.sum()),
            'HR_median': context['HR_median'], 'HR_q025': context['HR_q025'],
            'HR_q975': context['HR_q975'], 'joint_oriented_z': joint_z,
            'joint_permutation_p': joint_p,
        }
        for metric_index, metric in enumerate(METRICS):
            row[f'contrast__{metric}'] = observed[metric_index]
            row[f'z__{metric}'] = z[metric_index]
            row[f'p__{metric}'] = p[metric_index]

        context_name = safe_name(context['context'])
        pd.DataFrame(adverse, index=graph_features, columns=graph_features).to_csv(
            representation_matrix_dir / f'{context_name}__dataset_adjusted__adverse.csv'
        )
        pd.DataFrame(favorable, index=graph_features, columns=graph_features).to_csv(
            representation_matrix_dir / f'{context_name}__dataset_adjusted__favorable.csv'
        )
        np.savez_compressed(null_root / f'{representation}__{context_name}.npz', null=null, joint_null=joint_null)
        save_permutation_diagnostics(
            diagnostic_root / f'{representation}__{context_name}.npz', permutation_diagnostics
        )
        permutation_summary_row = diagnostic_summary(
            'pooled_membership_permutations', representation, context['context'],
            permutation_diagnostics,
        )
        checkpoint = {
            'signature': signature,
            'result': row, 'observed_diagnostics': observed_diagnostics,
            'permutation_diagnostic_summary': permutation_summary_row,
        }
        tmp = result_path.with_suffix('.tmp.json')
        tmp.write_text(json.dumps(checkpoint, indent=2))
        tmp.replace(result_path)
        partial.unlink(missing_ok=True)
        current = [json.loads(p.read_text())['result'] for p in sorted(OUTPUT_DIR.glob('checkpoint__*.json'))]
        pd.DataFrame(current).to_csv(OUTPUT_DIR/'completed_context_results.csv',index=False)
        result_rows.append(row)
        observed_diagnostic_rows.extend(observed_diagnostics)
        permutation_diagnostic_rows.append(permutation_summary_row)
        print(
            f"[{context_number}/{len(contexts)}] {context['context']} | "
            f'joint z={joint_z:.2f}, p={joint_p:.4g}', flush=True
        )

results = pd.DataFrame(result_rows)
results['joint_rank'] = results.groupby('field').joint_oriented_z.rank(ascending=False, method='min').astype(int)
for field, index in results.groupby('field').groups.items():
    index = list(index)
    results.loc[index, 'joint_FDR'] = benjamini_hochberg(
        results.loc[index, 'joint_permutation_p']
    )
    for metric in METRICS:
        results.loc[index, f'FDR__{metric}'] = benjamini_hochberg(
            results.loc[index, f'p__{metric}']
        )
results.to_csv(OUTPUT_DIR / 'mean_predictive_field_context_results.csv', index=False)
pd.DataFrame(observed_diagnostic_rows).to_csv(OUTPUT_DIR / 'observed_mean_matrix_psd_diagnostics.csv', index=False)
pd.DataFrame(permutation_diagnostic_rows).to_csv(OUTPUT_DIR / 'permutation_mean_matrix_psd_summary.csv', index=False)
bundle_inventory = []
for path in sorted(bundle_root.rglob('*.npz')):
    archive = np.load(path, allow_pickle=False)
    metadata = json.loads(str(archive['metadata_json']))
    bundle_inventory.append({
        **metadata, 'bundle_path': str(path),
        'features': len(archive['feature_names']),
        'correlation_draws': archive['adverse_feature_sd_by_draw'].shape[0],
        'removed_coordinates': ';'.join(archive['removed_coordinates'].astype(str)),
        'reproduction_inputs_path': str(archive['reproduction_inputs_path']),
    })
pd.DataFrame(bundle_inventory).to_csv(
    OUTPUT_DIR / 'compositional_sensitivity_bundle_inventory.csv', index=False
)
display(results.sort_values(['field', 'joint_rank']))